# Token-Set Audit on Cloud GPU (Kaggle / Colab)

Self-contained notebook for the multi-model POPE audit. Works on:

- **Kaggle**: enable T4 x 2 (or P100) in Settings -> Accelerator. Internet must be on.
- **Colab**: Runtime -> Change runtime type -> T4 GPU. Free tier is enough.
- **Local Jupyter**: any CUDA box.

Output JSONs land in `/kaggle/working/` (Kaggle) or `./outputs/` (Colab / local).
Pull them back with `scripts/pull_cloud_results.py` after the run finishes.

## Parallel strategy

Kaggle allows 2 concurrent GPU sessions per account. Free Colab allows 1.
Open several copies of this notebook with different `MODEL` values in the
config cell below; that gives you 3 parallel jobs (2 Kaggle + 1 Colab)
per Google account.

Per-model wall-clock on a single T4 (4-bit NF4, all 3 splits = 9000 q):

| Model                  | All 3 splits | One split | Notes                                |
|------------------------|-------------:|----------:|--------------------------------------|
| llava15                | ~80 min      | ~25 min   | reference, already done locally      |
| llava16_mistral        | ~110 min     | ~35 min   | LLaVA-1.6 Mistral                    |
| instructblip           | ~150 min     | ~50 min   | Vicuna-7B backend, larger Q-Former   |
| mplug_owl2             | ~180 min     | ~60 min   | needs trust_remote_code              |
| qwen2_vl               | ~140 min     | ~45 min   | requires transformers>=4.45          |

Stay under Kaggle's 12 h batch / 9 h interactive session limit by running
**one model at a time** in each notebook instance.

## 1. Configuration

In [ ]:
# Edit these three lines per session.
MODEL  = 'instructblip'                     # llava15 | llava16_mistral | instructblip | mplug_owl2 | qwen2_vl
SPLITS = ['adversarial', 'popular', 'random']
SAMPLES = 3000                              # 3000 = full split; reduce for a fast smoke test
PROMPT_MODES = ['paper_template']           # add 'native_chat_template' for a 2x run
REPO_URL = 'https://github.com/Kesav2k04/ugaa-research.git'
REPO_BRANCH = 'ugaa-v2'                     # branch to clone
QUANTIZE = '4bit'                           # 4bit | 8bit | none

import os, sys, pathlib
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
WORK = pathlib.Path('/kaggle/working') if IS_KAGGLE else (pathlib.Path('/content/work') if IS_COLAB else pathlib.Path.cwd() / 'cloud_run')
WORK.mkdir(parents=True, exist_ok=True)
print(f'Environment: kaggle={IS_KAGGLE} colab={IS_COLAB} work_dir={WORK}')

## 2. Install dependencies

Kaggle base images ship with `torch` and `transformers` already; we still
install our exact pins to keep the run reproducible. The `--quiet` flag
avoids the 30+ MB of pip log noise.

In [ ]:
REQUIRED_TRANSFORMERS = '4.45.2' if MODEL == 'qwen2_vl' else '4.40.1'
import subprocess, sys
def pip(*args):
    print('pip', *args)
    return subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])

pip(f'transformers=={REQUIRED_TRANSFORMERS}')
pip('accelerate>=0.28,<0.40', 'bitsandbytes>=0.43,<0.45')
pip('sentencepiece>=0.2.0', 'protobuf>=3.20,<5.0', 'safetensors')
pip('pandas', 'pyarrow', 'requests', 'Pillow')
if MODEL == 'qwen2_vl':
    pip('qwen-vl-utils')
if MODEL == 'mplug_owl2':
    pip('einops')

## 3. Clone the repo and set CWD

In [ ]:
import subprocess, os
REPO_DIR = WORK / 'ugaa-research'
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth=1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--rebase'])
os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
subprocess.check_call(['ls', '-la'])

## 4. Download POPE data (3 parquet files + extract 9,000 images, ~400 MB)

In [ ]:
import subprocess
subprocess.check_call([sys.executable, 'scripts/download_pope_full.py', '--splits', *SPLITS])
subprocess.check_call(['ls', '-la', 'datasets/pope/'])

## 5. Set HF cache to /tmp (faster + freed when session ends)

In [ ]:
import os
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache/transformers'
os.environ['HF_DATASETS_CACHE'] = '/tmp/hf_cache/datasets'
os.makedirs('/tmp/hf_cache', exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

## 6. Run the multi-model audit

Loops over splits and prompt modes. Outputs land in
`experiments/multi_model/<model>_pope_<split>_<n>q_<prompt_mode>_token_audit.json`.

In [ ]:
import subprocess, time
RUNS_OK, RUNS_FAIL = [], []
for split in SPLITS:
    for mode in PROMPT_MODES:
        t0 = time.time()
        cmd = [
            sys.executable, 'src/run_multi_model_audit.py',
            '--model', MODEL,
            '--split', split,
            '--samples', str(SAMPLES),
            '--data-dir', 'datasets/pope',
            '--output-dir', 'experiments/multi_model',
            '--device', 'cuda',
            '--quantize', QUANTIZE,
            '--cache-dir', '/tmp/hf_cache',
            '--prompt-mode', mode,
        ]
        print('\n' + '=' * 78)
        print(' '.join(cmd))
        print('=' * 78)
        try:
            subprocess.check_call(cmd)
            RUNS_OK.append((split, mode, round(time.time() - t0, 1)))
        except subprocess.CalledProcessError as exc:
            RUNS_FAIL.append((split, mode, str(exc)))
            print(f'FAILED on split={split} mode={mode}: {exc}')
print('\nSummary:')
for r in RUNS_OK:   print(f'  OK  {r[0]:12s} {r[1]:22s} {r[2]:.1f}s')
for r in RUNS_FAIL: print(f'  BAD {r[0]:12s} {r[1]:22s} {r[2]}')

## 7. Copy outputs to a downloadable location

On Kaggle, `/kaggle/working/` shows up under the notebook's Output tab.
On Colab, the `outputs/` folder will appear in the file browser; right-click
any file to download. Alternative on Colab: mount Google Drive and copy.

In [ ]:
import shutil
DEST = pathlib.Path('/kaggle/working/outputs') if IS_KAGGLE else (WORK / 'outputs')
DEST.mkdir(parents=True, exist_ok=True)
src = pathlib.Path('experiments/multi_model')
n = 0
if src.exists():
    for f in src.iterdir():
        if f.is_file():
            shutil.copy2(f, DEST / f.name)
            n += 1
print(f'Copied {n} files to {DEST}')
for f in sorted(DEST.iterdir()):
    print(' ', f.name, f.stat().st_size, 'bytes')

## 8. Quick smoke-test of the new artifacts

Confirms each JSON parses and shows the four-readout summary for each run.

In [ ]:
import json, pathlib
for j in sorted(pathlib.Path('experiments/multi_model').glob('*.json')):
    with open(j) as f:
        d = json.load(f)
    if 'summary' not in d:
        print(j.name, '-> no summary')
        continue
    s = d['summary']
    print(f'\n{j.name}')
    print(f"  tokenizer  = {s.get('tokenizer_class')}")
    print(f"  dynamic_yes_ids = {s.get('dynamic_yes_ids')}")
    print(f"  dynamic_no_ids  = {s.get('dynamic_no_ids')}")
    for k in ('eval_legacy_2tok', 'eval_legacy_8tok', 'eval_dynamic_single', 'eval_string_parse'):
        e = s.get(k, {})
        print(f"  {k:22s} F1={e.get('f1')}  P={e.get('precision')}  R={e.get('recall')}  yr={e.get('yes_rate')}  unk={e.get('unknown')}")

## 9. Done

When the Output cell is finalised, download the JSONs from the Kaggle
notebook output panel (or your Colab file browser) and place them in
`experiments/multi_model/` in your local repo. Then on the local machine:

```bash
python scripts/pull_cloud_results.py --dir <downloaded_folder>
python analysis/fact_audit.py
python analysis/string_parse_equivalence.py
```

The fact_audit will pick up the new JSONs and Phase 5 will confirm whether
the artifact reproduces on the new model (look for the equivalence rate
lines).